# Validation Test (mesh study) - Capillary Rise (benchmark test case within SFB 1194)

This notebook and the corresponding startUp and evaluation notebook (CapillaryRise_SFB1194_runStartUp.ipynb, CapillaryRise_SFB1194_Evaluation.ipynb) are part of published results (cf. 4.5) found in *A comparative study of transient capillary rise using direct numerical simulations* (https://doi.org/10.1016/j.apm.2020.04.020). 

In [ ]:
#r "BoSSSpad.dll"
//#r "..\..\src\L4-application\BoSSSpad\bin\Release\net8.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [2]:
ExecutionQueues

index,type,value
0,BoSSS.Application.BoSSSpad.MiniBatchProcessorClient,MiniBatchProcessor client LocalPC @C:\BoSSS-localJobs\binaries
1,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries"


In [3]:
string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
userName

FDY\smuda

In [4]:
//var myBatch = ExecutionQueues[1];
var myBatch = (userName.Equals(@"FDY\jenkinsci")) ? GetDefaultQueue() : ExecutionQueues[1];
myBatch

RuntimeLocation,win\amd64
AdditionalEnvironmentVars,[ ]
DeploymentBaseDirectory,\\dc3\userspace\smuda\hpccluster\binaries
DeployRuntime,True
Name,FDY-WindowsHPC
DotnetRuntime,dotnet
Username,FDY\smuda
NumOfServiceCoresPerMPIprocess,1
ServerName,DC3
ComputeNodes,<null>
DefaultJobPriority,Normal


In [5]:
wmg.Init("CapillaryRisePaper", myBatch);

Project name is set to 'CapillaryRisePaper'.
Opening existing database '\\dc3\userspace\smuda\hpccluster\CapillaryRisePaper'.


In [6]:
wmg.SetNameBasedSessionJobControlCorrelation();

In [7]:
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

In [ ]:
bool runShortTests = true; // set to 'false' to run all test cases for the whole period of time (up tp 0.7), otherwise they will run only up to 0.1

## Sessions

In [8]:
static var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions

In [ ]:
// foreach (var s in sessions) {
//     if(s.Name.Contains("CapillaryRise_Omega1") && s.Name.Contains("meshStudy") && !s.Name.Contains("startUp") && s.CreationTime.Date == new DateTime(2025, 12, 2)) {
//         s.Delete(true);
//         //Console.WriteLine($"Deleted session from {s.CreationTime.Date}");
//     }
// }

In [ ]:
// static var sessions = BoSSSshell.WorkflowMgm.Sessions;
// sessions

## Physical settings

In [9]:
string[] setupS = new string[] {
    "Omega0.1", "Omega0.5", "Omega1", "Omega10", "Omega100"
};

In [10]:
static double R = 5e-3;

In [11]:
public class PhysicalSettings {

    public double rhoA;
    public double muA;
    public double rhoB;
    public double muB;
    public double sigma;

    public double betaL;
    public double theta_e;

    public Formula GravityValue;

    public PhysicalSettings(string setup) {

        muA = 0.01; 
        muB = 0.01 / 1000;

        betaL = 0;
        theta_e = 3.0 * Math.PI / 18.0;

        switch(setup) {
            case "Omega01": {

                rhoA = 1663.8;
                rhoB = 1663.8 / 1000;      
                sigma = 0.2;       // kg / s^2

                GravityValue = new Formula(
                    "GravY",
                    false,
                    "double GravY(double[] X) { return -1.04; } "
                );

                break;
            }
            case "Omega05": {

                rhoA = 133.0;
                rhoB = 133.0 / 1000;      
                sigma = 0.1;       // kg / s^2

                GravityValue = new Formula(
                    "GravY",
                    false,
                    "double GravY(double[] X) { return -6.51; } "
                );

                break;
            }
            case "Omega1": {

                rhoA = 83.1;
                rhoB = 83.1 / 1000;      
                sigma = 0.04;       // kg / s^2

                GravityValue = new Formula(
                    "GravY",
                    false,
                    "double GravY(double[] X) { return -4.17; } "
                );

                break;
            }
            case "Omega10": {

                rhoA = 3.3255;
                rhoB = 3.3255 / 1000;      
                sigma = 0.01;       // kg / s^2

                GravityValue = new Formula(
                    "GravY",
                    false,
                    "double GravY(double[] X) { return -26.042; } "
                );

                break;
            }
            case "Omega100": {

                rhoA = 0.33255;
                rhoB = 0.33255 / 1000;      
                sigma = 0.001;       // kg / s^2

                GravityValue = new Formula(
                    "GravY",
                    false,
                    "double GravY(double[] X) { return -26.042; } "
                );

                break;
            }
        }
    }
}

## Control settings

In [ ]:
// origin database including all sessions, in case of missing sessions in wmg, due to rerunning only a subset of all needed runs (reduced computational cost)
static var originDB = OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\bkup-2025Nov26_000000.CapillaryRisePaper");

In [13]:
static int[] Resolutions = new int[] { 1, 2, 4, 8, 16 };    // cells per radius R { 1, 2, 4, 8, 16 };

In [ ]:
public class StudySettings {

    public string studyName;

    public string OmegaCase;
    public PhysicalSettings PhysicalParams;

    public Guid restartID;
    public ISessionInfo restartSession;
    public TimestepNumber restartTimestep;

    public int meshRes;
    public int AMRlvl;
    public double hmin;

    public double t_end;
    public double dt;
    public int savePeriod;
    public int logPeriod;

    public StudySettings(string name, string testcase, string restartName, int mesh, int AMRlevel, double tend, double timeStep, int savePer = 100, int logPer = 100) {
        studyName = name;
        OmegaCase = testcase;
        PhysicalParams = new PhysicalSettings(testcase);

        var rSess = sessions.Where(s => s.Name.ToLower() == restartName.ToLower());
        //ISessionInfo restartSession;
        if (rSess.Count() == 1) {
            restartSession = rSess.Single();           
        } else if (rSess.Count() == 0) {
            // try to get from originDB
            var originSess = originDB.Sessions.Where(s => s.Name.ToLower() == restartName.ToLower());
            int originCount = originSess.Count();
            if (originCount == 1) {
                Console.WriteLine("StartUp session found in originDB!");    
                restartSession = originSess.Single();
            } else{
                throw new ApplicationException($"No startUp session found with name {restartName} for restart.");
            }
        } else {
            throw new ApplicationException($"No unique startUp session found with name {restartName} for restart.");
        }
        restartID = restartSession.ID;
      
        var lastTimestep = restartSession.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber).Last();
        restartTimestep = lastTimestep.TimeStepNumber;
        t_end = lastTimestep.PhysicalTime + tend;
        dt = timeStep;

        meshRes = mesh;
        AMRlvl = AMRlevel;  
        hmin = lastTimestep.GetField("Phi").Basis.GridDat.iGeomCells.h_min.Min();
        if (restartName.Contains("_AMR1_")) {
            hmin *= 2.0;
        }
        hmin /= Math.Pow(2, AMRlvl); 

        double dt_sigma = BoSSS.Solution.XNSECommon.XNSEUtils.GetCapillaryTimeStep(PhysicalParams.rhoA, PhysicalParams.rhoB, PhysicalParams.sigma, hmin, 2+1);
        Console.WriteLine($"  dt (user) = {timeStep}, dt (capillary) = {dt_sigma}");

        savePeriod = savePer;   
        logPeriod = logPer;

    }

}

In [ ]:
List<StudySettings> allStudies = new List<StudySettings>();
// convergence study    
allStudies.Add(new StudySettings("meshStudy", "Omega1", "CapillaryRise_Omega1_meshStudy_mesh1_StartUp", 1, 0, 0.7, 1e-5));    //mesh1
allStudies.Add(new StudySettings("meshStudy", "Omega1", "CapillaryRise_Omega1_meshStudy_mesh2_StartUp", 2, 0, 0.7, 1e-5));    //mesh2
allStudies.Add(new StudySettings("meshStudy", "Omega1", "CapillaryRise_Omega1_meshStudy_mesh4_StartUp", 4, 0, 0.7, 1e-5));    //mesh4
allStudies.Add(new StudySettings("meshStudy", "Omega1", "CapillaryRise_Omega1_meshStudy_mesh8_StartUp", 8, 0, 0.7, 1e-5));    //mesh8
//allStudies.Add(new StudySettings("meshStudy", "Omega1", "CapillaryRise_Omega1_meshStudy_mesh16_StartUp", 16, 0, 0.7, 1e-5));    //mesh16
//allStudies.Add(new StudySettings("meshStudy", "Omega1", "CapillaryRise_Omega1_meshStudy_mesh8_StartUp", 8, 1, 0.7, 1e-5));    //mesh8 + AMR1
//allStudies.Add(new StudySettings("meshStudy", "Omega1", "CapillaryRise_Omega1_meshStudy_mesh8_StartUp", 8, 2, 0.7, 1e-5));    //mesh8 + AMR2

StartUp session found in originDB!
  dt (user) = 1E-05, dt (capillary) = 0.0008040122028911148
StartUp session found in originDB!
  dt (user) = 1E-05, dt (capillary) = 0.00028426124041052076
StartUp session found in originDB!
  dt (user) = 1E-05, dt (capillary) = 0.00010050152536138935
StartUp session found in originDB!
  dt (user) = 1E-05, dt (capillary) = 3.553265505131399E-05


### check initial condition

In [16]:
public static (double[] x, double[] y) GetTerminalInterfaceShape_XYvalues(ISessionInfo evalSess) {

    var terminalStep = evalSess.Timesteps.WithoutSubSteps().OrderBy(ts => ts.TimeStepNumber.MajorNumber).Last();
    //Console.WriteLine($"Terminal time step: {terminalStep.PhysicalTime}");
    DGField phi = terminalStep.GetField("Phi");
    LevelSet LevSet = new LevelSet(phi.Basis, "LevelSet"); 
    LevSet.Acc(1.0, phi);           
    LevelSetTracker LsTrk = new LevelSetTracker((BoSSS.Foundation.Grid.Classic.GridData) phi.GridDat, CutCellQuadratureMethod.Saye, 1, new string[] { "A", "B" }, LevSet);
    LsTrk.UpdateTracker(terminalStep.PhysicalTime);

    MultidimensionalArray interP = BoSSS.Solution.XNSECommon.XNSEUtils.GetInterfacePoints(LsTrk, LevSet, quadRuleOrderForNodeSet:10);
    double[] x = interP.ExtractSubArrayShallow(new int[] { -1, 0 }).To1DArray();
    double[] y = interP.ExtractSubArrayShallow(new int[] { -1, 1 }).To1DArray();

    return (x, y);
    
}

In [ ]:
List<ISessionInfo> restartSess = allStudies.Select(study => study.restartSession).ToList<ISessionInfo>();
//List<ISessionInfo> restartSess = sessions.Where(s => s.Name.Contains("meshStudy")).ToList<ISessionInfo>();
restartSess

#0: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh8_startUp*	11/26/2025 3:02:33 AM	4e9f4fa4...
#1: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh4_startUp	11/24/2025 3:08:13 PM	0d08da88...
#2: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh2_startUp	11/24/2025 3:08:05 PM	ca56d38d...
#3: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh1_startUp	11/24/2025 3:07:56 PM	243a4381...
#4: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh16_startUp*	11/24/2025 3:08:32 PM	9bc27b0c...
#5: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh8_startUp*	11/24/2025 3:08:23 PM	52e81583...


In [37]:
public static Plot2Ddata GetTerminalShapes_Plot2Ddata(List<ISessionInfo> dataSess) {

    Plot2Ddata datShape =  new Plot2Ddata();

    int lcIdx = 1;
    //for(int i = 0; i < study.Length; i++) {
        foreach (var dataS in dataSess) {
            //if (dataS.Name.Contains(study[i])) {
                var shape = GetTerminalInterfaceShape_XYvalues(dataS);

                var fmt = new PlotFormat();
                fmt.Style = Styles.Lines;
                fmt.Style      = Styles.Points;
                fmt.PointType  = PointTypes.Circle;
                fmt.PointSize = 0.5;
                fmt.LineColor = (LineColors)(lcIdx);

                datShape.AddDataGroup((dataS.Name).Replace("_", "-"), shape.x, shape.y, fmt);
                lcIdx++;
            //}
        }
    //}

    // datShape.XrangeMin = -0.03;
    // datShape.XrangeMax = 0.03;
    // datShape.YrangeMin = 0.0;
    // datShape.YrangeMax = .01;

    datShape.ShowLegend = true;
    datShape.LegendAlignment = new string[] { "i", "t", "l"};

    return datShape;
}

In [48]:
GetTerminalShapes_Plot2Ddata(restartSess).ToGnuplot().PlotSVG(xRes:900,yRes:300)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.009 
 
 
 
 
 0.0095 
 
 
 
 
 0.01 
 
 
 
 
 0.0105 
 
 
 
 
 0.011 
 
 
 
 
 0.0115 
 
 
 
 
 0.012 
 
 
 
 
 0 
 
 
 
 
 0.0005 
 
 
 
 
 0.001 
 
 
 
 
 0.0015 
 
 
 
 
 0.002 
 
 
 
 
 0.0025 
 
 
 
 
 0.003 
 
 
 
 
 0.0035 
 
 
 
 
 0.004 
 
 
 
 
 0.0045 
 
 
 
 
 0.005 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CapillaryRise-Omega1-meshStudy-mesh8-startUp 
 
 
 CapillaryRise-Omega1-meshStudy-mesh8-startUp 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CapillaryRise-Omega1-meshStudy-mesh4-startUp 
 
 
 CapillaryRise-Omega1-meshStudy-mesh4-startUp 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CapillaryRise-Omega1-meshStudy-mesh2-startUp 
 
 
 CapillaryRise-Omega1-meshStudy-mesh2-startUp 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CapillaryRise-Omega1-meshStudy-mesh1-startUp 
 
 
 CapillaryRise-Omega1-meshStudy-mesh1-startUp 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CapillaryRise-Omega1-meshStudy-mesh16-startUp 
 
 
 CapillaryRise-Omega1-meshStudy-mesh16-startUp 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 

In [49]:
foreach (var rSess in restartSess) {
    
    var ts = rSess.Timesteps.WithoutSubSteps().OrderBy(ts => ts.TimeStepNumber.MajorNumber).Last();

    double l_time = ts.PhysicalTime;
                
    DGField phi = ts.GetField("Phi");
    LevelSet LevSet = new LevelSet(phi.Basis, "LevelSet"); 
    LevSet.Acc(1.0, phi);           
    LevelSetTracker LsTrk = new LevelSetTracker((BoSSS.Foundation.Grid.Classic.GridData) phi.GridDat, CutCellQuadratureMethod.Saye, 1, new string[] { "A", "B" }, LevSet);
    LsTrk.UpdateTracker(ts.PhysicalTime);  
         
    XDGField VelX = (XDGField)ts.GetField("VelocityX");
    XDGField VelY = (XDGField)ts.GetField("VelocityY");
    XDGField[] Velocity = new XDGField[] {VelX, VelY};  

    // get spurious velocities 
    //{
        double spuriousVelocity = 0.0;                        

        int momentFittingOrder  = 6;
        var SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), momentFittingOrder, 1).XQuadSchemeHelper;
                                    
        for(int iSpc = 0; iSpc < LsTrk.SpeciesIdS.Count; iSpc++) {
            SpeciesId spId = LsTrk.SpeciesIdS[iSpc];

            var scheme = SchemeHelper.GetVolumeQuadScheme(spId);

            int D = LsTrk.GridDat.SpatialDimension;
            for(int d = 0; d < D; d++) {
                DGField U = Velocity.ElementAt(d);
                spuriousVelocity += U.L2Error(null, momentFittingOrder, scheme);
            
            }
        } 
    //} 
 
    // get surface divergence
    //{
        ConventionalDGField[] meanVelocity = XNSEUtils.GetMeanVelocity(Velocity, LsTrk, 1.0, 1.0);
        double surfaceDivergence = EnergyUtils.GetSurfaceChangerate(LsTrk, meanVelocity, 6);   
    //} 

    Console.WriteLine($"Session: {rSess.Name} (Time: {l_time}):");
    Console.WriteLine($"    spuriousVelocity: {spuriousVelocity}, surfaceDivergence: {surfaceDivergence}");

}

Session: CapillaryRise_Omega1_meshStudy_mesh8_startUp (Time: 0.25999999999997847):
    spuriousVelocity: 4.200604207521136E-05, surfaceDivergence: 0.003912837411443534
Session: CapillaryRise_Omega1_meshStudy_mesh4_startUp (Time: 0.20000000000000367):
    spuriousVelocity: 1.3410156838318725E-05, surfaceDivergence: 9.441075357580617E-05
Session: CapillaryRise_Omega1_meshStudy_mesh2_startUp (Time: 0.20000000000000015):
    spuriousVelocity: 3.1032232242164014E-05, surfaceDivergence: 0.0007376064908639806
Session: CapillaryRise_Omega1_meshStudy_mesh1_startUp (Time: 0.20000000000000015):
    spuriousVelocity: 0.00023938126666504388, surfaceDivergence: -0.004269965126387698
Session: CapillaryRise_Omega1_meshStudy_mesh16_startUp (Time: 0.06600000000000711):
    spuriousVelocity: 3.4304800337096566E-05, surfaceDivergence: 0.0021452665313555696
Session: CapillaryRise_Omega1_meshStudy_mesh8_startUp (Time: 0.27399999999999247):
    spuriousVelocity: 4.1841123743182125E-05, surfaceDivergence: 0.0

In [17]:
string[] slipCases = new string[] {
    "resolvedSlipR5", "resolvedSlipR50", "effectiveSlip"
};

In [18]:
public static double GetSlipLength(string slip, double hmin) {
    switch(slip) {
        case "resolvedSlipR5": 
            return R / 5.0;
        case "resolvedSlipR50":
            return R / 50.0;
            case "effectiveSlip":
            return hmin;
        default:
            throw new ApplicationException($"No valid slip case {slip} found.");
        }
}

In [19]:
bool useSymmetry = true;

In [30]:
List<XNSE_Control> Controls = new List<XNSE_Control>();

In [ ]:
foreach(var study in allStudies) {
foreach(var slip in slipCases) {

if (study.studyName == "OmegaStudy" && slip != "resolvedSlipR5")
    continue;

var C = new XNSE_Control();

int k = 2;
C.SetDGdegree(k);

C.FieldOptions.Add(VariableNames.GravityY + "#A", new FieldOpts() {
    SaveToDB = FieldOpts.SaveToDBOpt.TRUE
});
C.FieldOptions.Add(VariableNames.GravityY + "#B", new FieldOpts() {
    SaveToDB = FieldOpts.SaveToDBOpt.TRUE
});

// physical parameters
// ===================
var pParams = study.PhysicalParams;

C.PhysicalParameters.rho_A = pParams.rhoA;
C.PhysicalParameters.mu_A = pParams.muA;
C.PhysicalParameters.rho_B = pParams.rhoB;
C.PhysicalParameters.mu_B = pParams.muB;

C.PhysicalParameters.Sigma = pParams.sigma;

double Lslip = GetSlipLength(slip, study.hmin);
C.PhysicalParameters.betaS_A = pParams.muA / Lslip;
C.PhysicalParameters.betaS_B = pParams.muB / Lslip;

C.PhysicalParameters.betaL = pParams.betaL;
C.PhysicalParameters.theta_e = pParams.theta_e;

C.PhysicalParameters.IncludeConvection = true;
C.PhysicalParameters.Material = true;

    
// set restart info
// =================
C.RestartInfo = Tuple.Create(study.restartID, study.restartTimestep);


// boundary conditions
// ===================
//C.AddBoundaryValue("wall_lower");
C.ChangeBoundaryCondition("wall_lower", "pressure_outlet_lower");
C.AddBoundaryValue("pressure_outlet_lower");
C.AddBoundaryValue("pressure_outlet_upper");

if(useSymmetry)
    C.AddBoundaryValue("slipsymmetry_left");
else
    C.AddBoundaryValue("navierslip_linear_left");

C.AddBoundaryValue("navierslip_linear_right");

C.AdvancedDiscretizationOptions.GNBC_Localization = NavierSlip_Localization.Bulk;
C.AdvancedDiscretizationOptions.GNBC_SlipLength = NavierSlip_SlipLength.Prescribed_SlipLength;
C.PhysicalParameters.sliplength = Lslip;


// misc. solver options
// ====================
C.AdvancedDiscretizationOptions.SST_isotropicMode = SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;

C.CutCellQuadratureType = CutCellQuadratureMethod.Saye;
C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;

C.LinearSolver = LinearSolverCode.direct_mumps.GetConfig(); 
C.NonLinearSolver.SolverCode = NonLinearSolverCode.Picard;
C.NonLinearSolver.ConvergenceCriterion = 1e-9;
C.NonLinearSolver.MaxSolverIterations = 50;


// level-set
// =========
C.Option_LevelSetEvolution = LevelSetEvolution.FastMarching;

C.AdaptiveMeshRefinement = false;

if (study.AMRlvl > 0) {
    C.AdaptiveMeshRefinement = true;
    C.activeAMRlevelIndicators.Add(new AMRonNarrowbandAtBoundary(new byte[] { 4 }) { useUnion = true, maxRefinementLevel = study.AMRlvl });
    C.AMR_startUpSweeps = study.AMRlvl;
}

// Timestepping
// ============
C.TimeSteppingScheme = TimeSteppingScheme.BDF2;
C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;

C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
C.dtMin = study.dt;
C.dtMax = study.dt;
C.Endtime = runShortTests ? study.restartSession.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber).Last().PhysicalTime + 0.1 : study.t_end;
C.NoOfTimesteps = (int)(C.Endtime / C.dtMin);
C.saveperiod = study.savePeriod;
     
    
C.PostprocessingModules.Add(new BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases.MovingContactLineLogging() { SolverStage = 2, LogPeriod = study.logPeriod });
    
if (study.AMRlvl > 0) {
    C.SessionName = $"CapillaryRise_{study.OmegaCase}_{slip}_{study.studyName}_mesh{study.meshRes}_AMR{study.AMRlvl}";    
} else {
    C.SessionName = $"CapillaryRise_{study.OmegaCase}_{slip}_{study.studyName}_mesh{study.meshRes}";  
}
    
Controls.Add(C);

}
}

In [32]:
foreach (var ctrl in Controls) {
    Console.WriteLine($"{ctrl.SessionName}");
    // Console.WriteLine($"{ctrl.SessionName}: dt = {ctrl.dtMin}; saveperiod = {ctrl.saveperiod}; logperiod = {ctrl.PostprocessingModules.Pick(0).LogPeriod}");
}

CapillaryRise_Omega1_resolvedSlipR5_meshStudy_mesh1
CapillaryRise_Omega1_resolvedSlipR50_meshStudy_mesh1
CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh1
CapillaryRise_Omega1_resolvedSlipR5_meshStudy_mesh2
CapillaryRise_Omega1_resolvedSlipR50_meshStudy_mesh2
CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh2
CapillaryRise_Omega1_resolvedSlipR5_meshStudy_mesh4
CapillaryRise_Omega1_resolvedSlipR50_meshStudy_mesh4
CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh4
CapillaryRise_Omega1_resolvedSlipR5_meshStudy_mesh8
CapillaryRise_Omega1_resolvedSlipR50_meshStudy_mesh8
CapillaryRise_Omega1_effectiveSlip_meshStudy_mesh8
CapillaryRise_Omega05_resolvedSlipR5_OmegaStudy_mesh8_AMR1
CapillaryRise_Omega1_resolvedSlipR5_OmegaStudy_mesh8
CapillaryRise_Omega10_resolvedSlipR5_OmegaStudy_mesh8


In [33]:
int NoCtrls = Controls.Count();
NoCtrls

15

## Launch job

In [ ]:
List<Job> jobs = new List<Job>();

In [ ]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();
    
    oneJob.NumberOfMPIProcs = 1;

    //System.Environment.SetEnvironmentVariable("OMP_NUM_THREADS", "2");
    int numThreads = 1;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

    //oneJob.UseComputeNodesExclusive = true;\n",

    //((SlurmClient)myBatch).ExecutionTime = "24:00:00";

    oneJob.RetryCount = 1;
    oneJob.Activate(myBatch);

    jobs.Add(oneJob);
}

Deployments so far (1): (Job token: unknown, FinishedSuccessful 'unkown_DeploymentDirectory' @ MS HPC client  FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries, FinishedSuccessful);
Success: 1
Info: Found successful session "XNSEpaper_CapillaryWave	CapillaryWave_La3_convStudy_k2_mesh8	3/23/2020 9:35:37 PM	bdd58f86..." -- job is marked as successful, no further action.
No submission, because job status is: FinishedSuccessful
Deployments so far (1): (Job token: unknown, FinishedSuccessful 'unkown_DeploymentDirectory' @ MS HPC client  FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries, FinishedSuccessful);
Success: 1
Info: Found successful session "XNSEpaper_CapillaryWave	CapillaryWave_La3_convStudy_k2_mesh16	3/24/2020 2:29:40 PM	eed03bcf..." -- job is marked as successful, no further action.
No submission, because job status is: FinishedSuccessful
Deployments so far (1): (Job token: unknown, FinishedSuccessful 'unkown_DeploymentDirectory' @ MS HPC client  FDY-W

## Wait for Completion and Check Job Status

In [ ]:
int NoInProgress = jobs.Where(job => job.Status == JobStatus.InProgress 
                                    || job.Status == JobStatus.PendingInExecutionQueue
                                    || job.Status == JobStatus.PreActivation).Count();
NoInProgress

0

In [ ]:
int maxDays = 10;
int waitingDays = 0;
while (NoInProgress > 0 && waitingDays < maxDays) {
    wmg.BlockUntilAllJobsTerminate(3600*24*1); // wait one day for the jobs to finish
    NoInProgress = jobs.Where(job => job.Status == JobStatus.InProgress).Count();
    waitingDays++;
}

In [ ]:
var FailedSess = jobs.Where(job => job.Status == JobStatus.FailedOrCanceled);
FailedSess

0

In [ ]:
int NoFailed = FailedSess.Count();
NoFailed

In [ ]:
var SuccessSess = jobs.Where(job => job.Status == JobStatus.FinishedSuccessful);
SuccessSess

In [ ]:
int NoSuccess = SuccessSess.Count();
NoSuccess

17

In [ ]:
// check for restart sessions
if (NoFailed > 0) {
    foreach (var job in jobs) {
        //var job = ctrl.GetJob();
        if (job.Status == JobStatus.FailedOrCanceled) {
            var studySess = sessions.Where(sess => sess.Name.Contains(job.Name));
            int studyCount = studySess.Count();

            if (studyCount > 1) {
                Console.WriteLine("Restart session found! Take last run");       
                bool success = studySess.Where(sess => sess.Name.EndsWith($"_restart{studyCount-1}")).Single().SuccessfulTermination;
                if (success) {
                    Console.WriteLine($"Restart session {job.Name}_restart{studyCount-1} was successful.");
                    NoFailed--;
                    NoSuccess++;
                } else {
                    Console.WriteLine($"Restart session {job.Name}_restart{studyCount-1} also failed.");
                }
            } else if (studyCount == 1) {
                Console.WriteLine("No restart session found. {job.Name} might to be restarted.");
            } else { // studyCount == 0
                throw new ApplicationException($"No session found for {job.Name}!"); // should not happen
            } 
        }
    }

}

In [ ]:
//NUnit.Framework.Assert.AreEqual(0, NoFailed, $"Some simulations failed.");

In [ ]:
//NUnit.Framework.Assert.AreEqual(NoCtrls, NoSuccess, $"Not all simulations finished successfully.");